## Лабораторная работа 4

### Салятов Сергей, Подосенов Андрей, M3337

В нашем датасете приведены данные о результатах вступительных экзаменов

#### Задание 1: Распределение суммарного балла

Предположим, что суммарный балл распределен нормально

Гипотезы:

$H_0:$ данные распределены нормально

$H_1:$ данные распределены НЕ нормально

Проверим это вручную с помощью метода согласия Пирсона ($\chi^2$)

In [24]:
import pandas as pd
import numpy as np
from scipy import stats

In [25]:
alpha = 0.05
df = pd.read_csv("exams_dataset.csv")

df['total_score'] = df['math score'] + df['reading score'] + df['writing score']
total_scores = df['total_score'].values
n = len(total_scores)

mu = np.mean(total_scores)
sigma = np.std(total_scores, ddof=1)

print(f"Параметры нормального распределения:")
print(f"Среднее (mu) = {mu:.4f}")
print(f"Стандартное отклонение (sigma) = {sigma:.4f}\n")

num_bins = 10
expected_freq = n / num_bins

quantiles = np.linspace(0, 1, num_bins + 1)
bin_edges = stats.norm.ppf(quantiles, loc=mu, scale=sigma)

observed_freq, _ = np.histogram(total_scores, bins=bin_edges)

chi_sq_stat = 0

for i in range(num_bins):
    O = observed_freq[i]
    E = expected_freq
    component = (O - E) ** 2 / E
    chi_sq_stat += component

df_chi = num_bins - 1 - 2
critical_value = stats.chi2.ppf(1 - alpha, df_chi)

print(f"Степени свободы: {df_chi}")
print(f"Критическое значение для alpha={alpha}: {critical_value:.4f}")
print(f"Статистика Хи-квадрат: {chi_sq_stat:.4f}\n")

if chi_sq_stat > critical_value:
    decision = "Отвергаем H0: распределение НЕ является нормальным"
else:
    decision = "Не отвергаем H0: распределение можно считать нормальным"

print("РЕЗУЛЬТАТ ТЕСТА:")
print(f"Решение: {decision}")
print(f"(Критерий: chi_sq_stat = {chi_sq_stat:.4f} {'>' if chi_sq_stat > critical_value else '<='} critical_value = {critical_value:.4f})")

p_value = 1 - stats.chi2.cdf(chi_sq_stat, df_chi)
print(f"p-value = {p_value:.4f}")
if p_value < alpha:
    print("p-value < alpha → отвергаем H0")
else:
    print("p-value >= alpha → не отвергаем H0")

Параметры нормального распределения:
Среднее (mu) = 202.4040
Стандартное отклонение (sigma) = 45.4738

Степени свободы: 7
Критическое значение для alpha=0.05: 14.0671
Статистика Хи-квадрат: 13.0200

РЕЗУЛЬТАТ ТЕСТА:
Решение: Не отвергаем H0: распределение можно считать нормальным
(Критерий: chi_sq_stat = 13.0200 <= critical_value = 14.0671)
p-value = 0.0716
p-value >= alpha → не отвергаем H0


Теперь проверим с помощью встроенного теста Лиллиефорса (модификация KS-тест Колмогорова, когда математическое ожидание и дисперсия неизвестны)

In [26]:
import statsmodels.stats.diagnostic as smsd
stat, p_value = smsd.lilliefors(total_scores, dist='norm')
print("РЕЗУЛЬТАТЫ ТЕСТА ЛИЛЛИЕФОРСА (модифицированный KS-тест):")
print(f"p-value = {p_value:.6f}")

if p_value < alpha:
    conclusion = "Отвергаем H0: распределение НЕ является нормальным (p < 0.05)"
else:
    conclusion = "Не отвергаем H0: нет оснований считать распределение ненормальным (p >= 0.05)"

print(conclusion)

РЕЗУЛЬТАТЫ ТЕСТА ЛИЛЛИЕФОРСА (модифицированный KS-тест):
p-value = 0.053289
Не отвергаем H0: нет оснований считать распределение ненормальным (p >= 0.05)


#### Задание 2: Одинаковость распределений баллов по математике и чтению

Предположим, что баллы по математике и чтению распределены одианково

Гипотезы:

$H_0:$ баллы по математике и чтению распределены одинаково ($F_X = F_Y$)

$H_1:$ баллы по математике и чтению распределены НЕ одинаково ($F_X \neq F_Y$)

Проверим это вручную с помощью метода Манна-Уитни

Так как размер выборки большой, то будем использовать нормальное распределение для $Z$-статистики ошибки (посчитав предварительно метаматическое ожидание и дисперсию)

$$
\mu = \frac{n_1 n_2}{2}
$$
$$
\sigma^2 = \frac{n_1 n_2 (n_1 + n_2 + 1)}{12}
$$

In [27]:
math_scores = df['math score'].values
reading_scores = df['reading score'].values

n1 = len(math_scores)
n2 = len(reading_scores)

print("="*60)
print("ТЕСТ 1: РУЧНОЙ РАСЧЁТ — КРИТЕРИЙ МАННА-УИТНИ (U-ТЕСТ)")
print("="*60)

combined_data = np.concatenate([math_scores, reading_scores])
group_labels = np.array(['math'] * n1 + ['reading'] * n2)

sort_indices = np.argsort(combined_data)
sorted_data = combined_data[sort_indices]
sorted_labels = group_labels[sort_indices]

value_ranks = {}
current_rank = 1
i = 0

while i < len(sorted_data):
    j = i
    while j < len(sorted_data) and sorted_data[j] == sorted_data[i]:
        j += 1
    
    tie_ranks = list(range(current_rank, current_rank + (j - i)))
    avg_rank = np.mean(tie_ranks)
    
    for k in range(i, j):
        value_ranks[sorted_data[k]] = avg_rank
    
    current_rank += (j - i)
    i = j

ranks = np.array([value_ranks[x] for x in combined_data])

math_ranks = ranks[group_labels == 'math']
reading_ranks = ranks[group_labels == 'reading']

R1 = np.sum(math_ranks)
R2 = np.sum(reading_ranks)

U1 = R1 - n1 * (n1 + 1) / 2
U2 = R2 - n2 * (n2 + 1) / 2
U = min(U1, U2)

print(f"\nU1 (математика): {U1:.2f}")
print(f"U2 (чтение): {U2:.2f}")
print(f"Статистика U = min(U1, U2) = {U:.2f}")

mu_U = n1 * n2 / 2
sigma_U = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

Z = (U - mu_U + 0.5) / sigma_U
Z_crit = stats.norm.ppf(1 - alpha / 2)

print(f"\nНормальная аппроксимация:")
print(f"mu_U = {mu_U:.2f}, sigma_U = {sigma_U:.2f}")
print(f"Z-статистика (с поправкой на непрерывность): {Z:.4f}")
print(f"Критическое значение Z-статистики для двусторонненго теста с alpha = 0.05: {Z_crit}")

p_value_manual = 2 * (1 - stats.norm.cdf(abs(Z)))
print(f"p-value (ручной расчёт): {p_value_manual:.6f}")

if p_value_manual < alpha:
    decision_manual = "Отвергаем H0: распределения различаются"
else:
    decision_manual = "Не отвергаем H0: нет оснований считать распределения разными"
print(f"\nРЕШЕНИЕ (p-value): {decision_manual}")

if Z_crit < abs(Z):
    decision_manual = "Отвергаем H0: распределения различаются"
else:
    decision_manual = "Не отвергаем H0: нет оснований считать распределения разными"
print(f"\nРЕШЕНИЕ (критическое значение): {decision_manual}")

ТЕСТ 1: РУЧНОЙ РАСЧЁТ — КРИТЕРИЙ МАННА-УИТНИ (U-ТЕСТ)

U1 (математика): 448792.00
U2 (чтение): 551208.00
Статистика U = min(U1, U2) = 448792.00

Нормальная аппроксимация:
mu_U = 500000.00, sigma_U = 12913.17
Z-статистика (с поправкой на непрерывность): -3.9655
Критическое значение Z-статистики для двусторонненго теста с alpha = 0.05: 1.959963984540054
p-value (ручной расчёт): 0.000073

РЕШЕНИЕ (p-value): Отвергаем H0: распределения различаются

РЕШЕНИЕ (критическое значение): Отвергаем H0: распределения различаются


Теперь проверим встроенным KS-тестом Смирнова

In [28]:
print("\n" + "="*60)
print("ТЕСТ 2: ВСТРОЕННАЯ РЕАЛИЗАЦИЯ — ДВУХВЫБОРОЧНЫЙ KS-ТЕСТ")
print("="*60)

ks_stat, p_value_ks = stats.ks_2samp(math_scores, reading_scores)

print(f"Статистика KS (D): {ks_stat:.4f}")
print(f"p-value: {p_value_ks:.6f}")

if p_value_ks < alpha:
    decision_ks = "Отвергаем H0: распределения различаются"
else:
    decision_ks = "Не отвергаем H0: нет оснований считать распределения разными"

print(f"РЕШЕНИЕ (KS-тест): {decision_ks}")


ТЕСТ 2: ВСТРОЕННАЯ РЕАЛИЗАЦИЯ — ДВУХВЫБОРОЧНЫЙ KS-ТЕСТ
Статистика KS (D): 0.0890
p-value: 0.000721
РЕШЕНИЕ (KS-тест): Отвергаем H0: распределения различаются


#### Задание 3: Верно ли, что прослушавшие подготовительные курсы лучше сдали экзамены?

Предположим, что суммарный балл распределен нормально

Гипотезы:

- $H_0:$ прохождение курсов не повлияло на результат сдачи экзаменов

- $H_1:$ студенты, прошедние подготовительные курсы, сдали экзамены лучше

Используемые тесты: 

- T-test для двух выборок

- Критерий Мана-Уитни


In [31]:
from scipy.stats import t, mannwhitneyu

group1 = df[df['test preparation course'] == 'completed']['total_score'].values
group2 = df[df['test preparation course'] == 'none']['total_score'].values

n1, n2 = len(group1), len(group2)
mean1, mean2 = np.mean(group1), np.mean(group2)
var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)

t_stat = (mean1 - mean2) / np.sqrt(var1 / n1 + var2 / n2)

df_welch = (var1 / n1 + var2 / n2) ** 2 / ((var1 / n1) ** 2 / (n1 - 1) + (var2 / n2) ** 2 / (n2 - 1))

alpha = 0.05
p_value = 1 - t.cdf(t_stat, df=df_welch)
t_crit = t.ppf(1 - alpha, df=df_welch)


print("t-статистика:", t_stat)
print("Степени свободы:", df_welch)
print("Критическое значение t при alpha=0.05:", t_crit)
print("p-value:", p_value)

if p_value < 0.05 or t_stat > t_crit:
    print("H0 отвергается: группа с курсами сдала лучше (статистически значимо).")
else:
    print("Нет оснований отвергать H0: статистически значимой разницы нет.")


# Mann–Whitney test

import numpy as np
from scipy.stats import mannwhitneyu, norm

n1 = len(group1)
n2 = len(group2)

u, p = mannwhitneyu(group1, group2, alternative='greater')

mu_U = n1 * n2 / 2
sigma_U = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

alpha = 0.05
z_crit = norm.ppf(1 - alpha)

u_crit = mu_U + z_crit * sigma_U

print("U-статистика:", u)
print("p-value:", p)
print("Критическое значение U для alpha=0.05:", u_crit)

if u <= u_crit or p >= 0.05:
    print("U-статистика меньше критического значения → H0 не отвергается")
    print("H1 не подтверждается: статистически значимой разницы нет.")
else:
    print("U-статистика больше критического значения → H0 отвергается")
    print("H1 принимается: прошедшие курсы сдали экзамены лучше.")


t-статистика: 8.559942204819432
Степени свободы: 810.9310162379883
Критическое значение t при alpha=0.05: 1.6467348251504443
p-value: 0.0
H0 отвергается: группа с курсами сдала лучше (статистически значимо).
U-статистика: 152265.0
p-value: 1.0672468653419847e-15
Критическое значение U для alpha=0.05: 124460.42844962666
U-статистика больше критического значения → H0 отвергается
H1 принимается: прошедшие курсы сдали экзамены лучше.
